# Week 7: Introductie GeoPandas in JupyterLite
In deze les leren we hoe we geografische data kunnen verwerken met **GeoPandas** direct in de browser via **JupyterLite**.

## Deel 1: Packages installeren met `piplite`
In JupyterLite installeren we `pandas`, `shapely` en `matplotlib`. Daarna installeren we `geopandas` zonder C-afhankelijkheden via `deps=False`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import piplite

# 1. Pure Python basispakketten installeren
await piplite.install(['pandas', 'shapely', 'matplotlib', 'folium'])

# 2. GeoPandas installeren zonder C-drivers
await piplite.install('geopandas', deps=False)

import geopandas as gpd
print("GeoPandas versie:", gpd.__version__)

## Deel 2: Handmatig een GeoDataFrame & CRS instellen
Een **GeoDataFrame** is een Pandas DataFrame met een speciale `geometry` kolom. We stellen het Coordinate Reference System (CRS) in op `EPSG:4326` (WGS84 - lengtegraden en breedtegraden).

> **Let op:** In JupyterLite gebruiken we `EPSG:4326` als standaard. Herprojectie via `.to_crs()` werkt niet in de browser omdat de gecompileerde PROJ C-bibliotheek ontbreekt.

In [ ]:
import pandas as pd
from shapely.geometry import Point

# Data met steden en hun coördinaten (Breedtegraad / Lengtegraad)
data = {
    "Stad": ["Amsterdam", "Utrecht", "Rotterdam", "Groningen"],
    "Inwoners": [872000, 360000, 650000, 233000],
    "Lat": [52.3676, 52.0907, 51.9244, 53.2194],
    "Lon": [4.9041, 5.1214, 4.4777, 6.5665]
}

df = pd.DataFrame(data)

# Shapely Points maken (let op de volgorde: Point(longitude, latitude))
geometry = [Point(lon, lat) for lon, lat in zip(df["Lon"], df["Lat"])]

# GeoDataFrame maken met WGS84 coördinatenstelsel (EPSG:4326)
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
gdf

## Deel 3: Kaart Visualiseren
Met de `.plot()` methode kun je de ruimtelijke punten op een kaart laten zien.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, color="red", markersize=gdf["Inwoners"]/10000, alpha=0.7)

# Labels toevoegen per stad
for x, y, label in zip(gdf.geometry.x, gdf.geometry.y, gdf["Stad"]):
    ax.text(x + 0.05, y, label, fontsize=10)

plt.title("Nederlandse Steden (Grootte = Inwoners)")
plt.xlabel("Lengtegraad (Longitude)")
plt.ylabel("Breedtegraad (Latitude)")
plt.grid(True)
plt.show()

## Deel 4: Afstanden berekenen in kilometers (Haversine)
Omdat we geen `.to_crs()` naar een meter-gebaseerd stelsel kunnen gebruiken, berekenen we de afstand over de aarde via de Haversine-formule.

In [ ]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Straal van de aarde in km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# Afstand berekenen vanaf Amsterdam (Lat: 52.3676, Lon: 4.9041)
ams_lat, ams_lon = 52.3676, 4.9041
gdf["Afstand_tot_Amsterdam_km"] = [round(haversine(ams_lat, ams_lon, lat, lon), 1) for lat, lon in zip(gdf["Lat"], gdf["Lon"])]

gdf[["Stad", "Afstand_tot_Amsterdam_km"]]

## Deel 5: Geodata Exporteren & Inlezen
We exporteren en importeren GeoJSON via de standaard Python `json` module.

In [ ]:
import json

# 1. Exporteren naar GeoJSON via .to_json()
geojson_string = gdf.to_json()
with open("steden.geojson", "w") as f:
    f.write(geojson_string)
print("Bestand 'steden.geojson' is opgeslagen!")

# 2. Inlezen vanuit GeoJSON via json.load() en GeoDataFrame.from_features()
with open("steden.geojson", "r") as f:
    geojson_data = json.load(f)

ingelezen_gdf = gpd.GeoDataFrame.from_features(geojson_data["features"], crs="EPSG:4326")
ingelezen_gdf

### Folium
We kunnen deze geojson data ook plotten op een kaart.

In [ ]:
import folium

m = folium.Map(location=[52.3676, 4.9041], zoom_start=7)
folium.GeoJson('steden.geojson', name='Steden').add_to(m)
m  # Rendert direct een interactieve kaart in de notebook!

### Uitgebreidere uitwerking met Folium

In [ ]:
import folium
import math

# 1. Haversine functie om afstanden te berekenen (in km)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Straal van de aarde
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# 2. Basiskaart maken (gecentreerd op Nederland)
m = folium.Map(location=[52.2, 5.3], zoom_start=8, tiles="OpenStreetMap")

# Coördinaten van Amsterdam als centraal punt
ams_lat, ams_lon = 52.3676, 4.9041

# 3. GeoJSON punten / markers toevoegen
for index, row in gdf.iterrows():
    lat, lon, stad = row["Lat"], row["Lon"], row["Stad"]
    
    # Marker toevoegen per stad
    folium.Marker(
        location=[lat, lon],
        popup=f"<b>{stad}</b><br>Inwoners: {row['Inwoners']:,}",
        tooltip=stad,
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(m)

    # 4. Lijnen en afstands-labels tekenen vanaf Amsterdam naar de andere steden
    if stad != "Amsterdam":
        afstand = round(haversine(ams_lat, ams_lon, lat, lon), 1)
        
        # Verbindingslijn tekenen tussen Amsterdam en stad
        line_coords = [[ams_lat, ams_lon], [lat, lon]]
        folium.PolyLine(
            line_coords,
            color="red",
            weight=2.5,
            opacity=0.8,
            tooltip=f"Afstand Amsterdam - {stad}: {afstand} km"
        ).add_to(m)
        
        # Middelpunt bepalen voor het afstandslabel op de lijn
        mid_lat = (ams_lat + lat) / 2
        mid_lon = (ams_lon + lon) / 2
        
        # Tekst-label halverwege de lijn plaatsen via DivIcon
        label_html = f"""
        <div style="
            font-size: 10pt; 
            font-weight: bold; 
            color: black; 
            ">{afstand} km
        </div>
        """
        folium.Marker(
            location=[mid_lat, mid_lon],
            icon=folium.DivIcon(html=label_html)
        ).add_to(m)

# Toon de interactieve kaart direct in het notebook
m

# Huiswerkopdracht: De 12 Grootste HBO-instellingen in Kaart (met GeoPandas)

In deze opdracht ga je de **12 grootste HBO-instellingen** van Nederland in kaart brengen. We combineren hierbij **Pandas**, **GeoPandas** en **Folium**.

### Doelstellingen van de opdracht:
1. **Van DataFrame naar GeoDataFrame**: Converteer de tabel met geografische coördinaten (`Lat`/`Lon`) naar een **GeoPandas GeoDataFrame** met `EPSG:4326` als CRS.
2. **GeoJSON Exporteren**: Sla het GeoDataFrame op als `hbo_instellingen.geojson` met `.to_json()` of `.to_file()`.
3. **Interactieve Kaart met `folium.GeoJson`**: Geef de data weer op een Folium-kaart. Zorg ervoor dat bij het bewegen over of klikken op een marker de volgende gegevens zichtbaar zijn:
   - **Naam van de Hogeschool**
   - **Stad / Vestiging**
   - **Aantal studenten**
   - **Aantal medewerkers**

---

### Startcode & Dataset
Controleer de start data. Voeg eventueel ontbrekende data toe. Corrigeer onjuiste data.

In [22]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium

# 1. Dataset van de 12 grootste HBO-instellingen (inclusief multi-locaties)
hbo_data = {
    "Instelling": [
        "Hogeschool van Amsterdam", "Fontys Hogeschool", "Fontys Hogeschool", "Fontys Hogeschool",
        "Hogeschool Utrecht", "Hogeschool Rotterdam", "Saxion Hogeschool", "Saxion Hogeschool",
        "HAN University of Applied Sciences", "HAN University of Applied Sciences",
        "Avans Hogeschool", "Avans Hogeschool", "Windesheim", "Windesheim",
        "Hogeschool Inholland", "Hogeschool Inholland", "Hogeschool Inholland",
        "De Haagse Hogeschool", "NHL Stenden Hogeschool", "NHL Stenden Hogeschool",
        "Zuyd Hogeschool", "Zuyd Hogeschool"
    ],
    "Stad": [
        "Amsterdam", "Eindhoven", "Tilburg", "Venlo",
        "Utrecht", "Rotterdam", "Enschede", "Deventer",
        "Arnhem", "Nijmegen",
        "Breda", "'s-Hertogenbosch", "Zwolle", "Almere",
        "Haarlem", "Delft", "Alkmaar",
        "Den Haag", "Leeuwarden", "Emmen",
        "Heerlen", "Sittard"
    ],
    "Lat": [
        52.3590, 51.4516, 51.5631, 51.3653,
        52.0842, 51.9172, 52.2226, 52.2570,
        51.9866, 51.8260,
        51.5835, 51.6978, 52.5034, 52.3702,
        52.3888, 52.0016, 52.6372,
        52.0672, 53.2118, 52.7781,
        50.8792, 51.0003
    ],
    "Lon": [
        4.9080, 5.4812, 5.0863, 6.1627,
        5.1742, 4.4842, 6.8951, 6.1620,
        5.9490, 5.8613,
        4.7972, 5.2837, 6.0921, 5.2141,
        4.6462, 4.3705, 4.7501,
        4.3235, 5.7958, 6.9026,
        5.9592, 5.8682
    ],
    "Studenten": [
        44000, 44000, 44000, 44000,
        38000, 38000, 27000, 27000,
        33000, 33000,
        31000, 31000, 27000, 27000,
        28000, 28000, 28000,
        26000, 24000, 24000,
        14000, 14000
    ],
    "Medewerkers": [
        4200, 4500, 4500, 4500,
        3700, 3600, 2800, 2800,
        3500, 3500,
        3000, 3000, 2600, 2600,
        2400, 2400, 2400,
        2500, 2200, 2200,
        1800, 1800
    ]
}

df_hbo = pd.DataFrame(hbo_data)

# --- OPDRACHT STAPPENPLAN ---
# STAP 1: Maak Shapely Points van Lon & Lat en maak een GeoDataFrame met crs="EPSG:4326"
# geometry = ...
# gdf_hbo = gpd.GeoDataFrame(...)

# STAP 2: Sla op als GeoJSON bestand
# ...

# STAP 3: Maak een Folium kaart gecentreerd op Nederland
# m = folium.Map(location=[52.2, 5.3], zoom_start=7)

# STAP 4: Voeg gdf_hbo toe met folium.GeoJson(), inclusief GeoJsonTooltip en GeoJsonPopup
# folium.GeoJson(...).add_to(m)

# STAP 5: Toon de kaart!
# m
